# [Step 1 - Groq cloud inference] Same open models, extreme speed, free tier

> **MLCourse - Agentic AI - Chat Models and Providers**

> Stage in the capstone: the generate stage - when the final RAG chatbot needs
> snappy answers without a GPU on the host machine, a fast hosted engine like Groq
> is exactly how production deployments get local-quality models at cloud speed.

## What you'll learn

- What Groq is and why its LPU hardware makes it the fastest way to run open models like Llama.
- The guarded key pattern every cloud notebook in this track uses to stay runnable without keys.
- How to stream tokens live from a cloud endpoint and measure the throughput yourself.
- How the same prompt behaves across temperatures when a 70B model writes the answers.
- A fair methodology for comparing cloud latency against your local Ollama setup.

In [1]:
# --- Standard library imports -------------------------------------------------
import os                # Reads environment variables populated by load_dotenv below.
import time              # Timing calls so we can quantify "fast" instead of hand-waving.
from pathlib import Path # Used to locate the track root folder.

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv  # Loads KEY=value pairs from .env into os.environ.

# Walk up to the track root and load 03_agentic_ai/.env (gitignored) - this is the
# shared key-loading block used by EVERY provider-touching notebook in the track.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Read the Groq key ONCE here; later cells check this variable before calling the cloud.
GROQ_KEY = os.getenv("GROQ_API_KEY")

# Jupyter plotting magic inside try/except so the file stays valid pure Python outside IPython.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Groq key present:", bool(GROQ_KEY))   # Prints True/False - never print the key itself!

Groq key present: False


## 1. The Groq story in ninety seconds

Groq (not to be confused with Google's TPU) built custom silicon called an **LPU**
for language-model inference. Because the hardware is specialized, Groq serves open
models such as `llama-3.3-70b-versatile` at hundreds of tokens per second - speeds
that would need a serious multi-GPU box to match locally. For learners this matters
twice over:

1. The **free tier** is generous enough to power this entire course: you only need a
   key from console.groq.com, pasted into `03_agentic_ai/.env` as `GROQ_API_KEY`.
2. Free tiers are **rate limited** (requests per minute plus tokens per day). Keep
   experiment outputs short and you will rarely hit the ceiling.

> **Common pitfall:** HTTP error 429 means you exceeded a rate limit. It is not a bug:
> wait a moment, shrink `max_tokens`, or switch to the smaller `llama-3.1-8b-instant`
> model, which has lighter limits.

> **Pro tip:** treat the key like a password. Read it with `os.getenv` and never print,
> commit, or screenshot it. The cell above printed only a boolean check.

## 2. Guarded first call

This is THE canonical pattern for cloud cells in this track: if the key is missing we
print setup instructions and skip; otherwise we build the client and call it. Notebooks
therefore execute green end-to-end for everyone, regardless of who has keys.

We use `llama-3.3-70b-versatile` - a 70-billion-parameter model most laptops could
never host - which is precisely why cloud access is interesting even for local-first
learners.

In [2]:
if not GROQ_KEY:
    # Missing key path: explain exactly how to fix it, then skip gracefully.
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env (see .env.example)")
else:
    from langchain_groq import ChatGroq

    llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_KEY)
    print(llm.invoke("Say hi in five words").content)

[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env (see .env.example)


## 3. Live token streaming from the cloud

Module-local notebook 01 streamed from Ollama; the exact same loop works against
Groq - one more proof that LangChain's interface hides the plumbing. Here we also
measure time-to-first-token versus total duration, because that split is what users
actually FEEL in a chat UI: first token arriving early reads as "fast" even if the
full answer takes the same total time.

> **Pro tip:** `max_tokens=120` caps the reply length. On metered or rate-limited
> providers, capping output is the single easiest quota-saving habit.

In [3]:
if not GROQ_KEY:
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env (see .env.example)")
else:
    from langchain_groq import ChatGroq

    llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_KEY, max_tokens=120)

    try:
        t0 = time.perf_counter()            # Start the stopwatch before the request.
        first_token_at = None               # We will stamp this when chunk number 1 lands.
        n_chunks = 0                        # Rough proxy for token count (chunks ~= tokens).
        for chunk in llm.stream("Explain retrieval-augmented generation in three sentences."):
            if first_token_at is None:      # First fragment just arrived - record the moment.
                first_token_at = time.perf_counter() - t0
            print(chunk.content, end="", flush=True)   # Print fragments as they stream in.
            n_chunks += 1
        total = time.perf_counter() - t0    # Full answer elapsed time.
        print()
        print(f"time to first token : {first_token_at:.2f}s")
        print(f"total time          : {total:.2f}s")
        print(f"approx throughput   : {n_chunks / total:.0f} chunks/second")
    except Exception:
        # Covers network hiccups AND rate limits without crashing the notebook run.
        print("[demo skipped] Groq call failed - check network access and rate limits.")

[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env (see .env.example)


## 4. One prompt, many temperatures

Notebook 01 ran the temperature experiment on a 3B model running locally; now watch a
70B model do it. Bigger models are usually *more* coherent at high temperatures - they
stay on-topic while varying phrasing more usefully than small models do.

We reuse ONE prompt string across all settings so temperature is the only variable -
change one thing at a time when you experiment.

In [4]:
PROMPT = "Give me a fun name for a study group of AI engineers, plus a five-word reason."

if not GROQ_KEY:
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env (see .env.example)")
else:
    from langchain_groq import ChatGroq

    for temp in (0.0, 0.7, 1.3):
        try:
            # Fresh client per setting: temperature is fixed at construction time.
            temp_llm = ChatGroq(
                model="llama-3.3-70b-versatile",
                api_key=GROQ_KEY,
                temperature=temp,
                max_tokens=80,           # Short outputs keep the free-tier quota healthy.
            )
            print(f"--- temperature = {temp} ---")
            print(temp_llm.invoke(PROMPT).content)
        except Exception:
            print("[demo skipped] Rate limit or network issue - wait a moment and rerun.")
            break                        # Stop the sweep rather than printing failures repeatedly.

[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env (see .env.example)


> **Common pitfall:** comparing providers by eyeballing one run. Latency varies with
> network conditions, server load, warm-up state, and model size. The honest method:
> same short prompt, several runs each, report rough averages - which is exactly what
> the next cell sets up.

In [5]:
SPEED_PROMPT = "In one short sentence, why does chunking matter for retrieval?"

def timed_invoke(make_llm, prompt):
    """Build a fresh model via factory function, invoke once, return (text, seconds)."""
    start = time.perf_counter()          # Clock starts AFTER construction so build time is excluded.
    text = make_llm().invoke(prompt).content
    return text, time.perf_counter() - start

results = {}                             # label -> (answer_text, seconds); collected for both engines.

# Cloud leg: only attempted when the key exists, wrapped against network/rate errors.
if GROQ_KEY:
    from langchain_groq import ChatGroq

    try:
        results["groq llama-3.3-70b"] = timed_invoke(
            lambda: ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_KEY, max_tokens=80),
            SPEED_PROMPT,
        )
    except Exception:
        print("[demo skipped] Groq unavailable right now.")

# Local leg: attempted unconditionally but guarded, since Ollama may not be installed.
from langchain_ollama import ChatOllama

try:
    results["local llama3.2"] = timed_invoke(
        lambda: ChatOllama(model="llama3.2"),
        SPEED_PROMPT,
    )
except Exception:
    print("[demo skipped] Start Ollama, then run once: ollama pull llama3.2")

for label, (text, seconds) in results.items():
    print(f"{label:<22} {seconds:5.2f}s | {text[:70]}")
print()
print("Caveat: different model sizes and one-shot timing - treat this as intuition, not a benchmark.")

[demo skipped] Start Ollama, then run once: ollama pull llama3.2

Caveat: different model sizes and one-shot timing - treat this as intuition, not a benchmark.


## Summary & key takeaways

- Groq runs open models on custom LPU hardware at extreme tokens/sec, with a free tier
  strong enough for the whole course - just mind the rate limits (HTTP 429).
- The **guard pattern** (`if not KEY: print instructions / else: call`) keeps cloud
  notebooks green for every student; expect it in every remaining module.
- Streaming works identically across providers; measuring time-to-first-token shows
  WHY streaming feels faster even when totals are similar.
- Temperature sweeps transfer directly from local to cloud - bigger models tend to
  stay coherent at higher creativity settings.
- Compare engines with repeated, identical, short prompts - and remember the capstone
  generate stage can sit on EITHER engine thanks to the abstraction coming in notebook 04.